# Document Library Chroma Query

In [ ]:
%reload_ext autoreload
%autoreload 2

from pathlib import Path
import sys
import os
from collections import Counter
from datetime import datetime, timezone


sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from apps.agentic.core.document_loaders.document_library_loader import DocumentLibraryLoader
from apps.agentic.core.constants import PDF_DOCUMENT_LIBRARY_DB_NAME, PDF_DOCUMENT_LIBRARY_COLLECTION_NAME
from apps.agentic.core.utils import (set_chatgpt_env, set_langsmith_env, set_tavily_env, 
                                     set_github_env)

DB_PATH = Path("../../.db").resolve()

set_chatgpt_env(filedir="../../.keys")
set_langsmith_env(filedir="../../.keys")

## Verify Document Counts

In [3]:
PDF_DOCUMENT_LIBRARY_DB_NAME, PDF_DOCUMENT_LIBRARY_COLLECTION_NAME, DB_PATH, os.listdir(DB_PATH)

('pdf_document_library',
 'pdf-documents',
 '../../.db',
 ['.DS_Store', 'research_library', 'pdf_document_library', 'github'])

In [4]:
vs = DocumentLibraryLoader(DB_PATH).vectorstore
coll = vs._collection

In [5]:
client = vs._client
print([c.name for c in client.list_collections()])
print("Opened:", coll.name)

['pdf-documents']
Opened: pdf-documents


In [8]:
print("total docs:", coll.count())

total docs: 30


In [9]:
res = coll.get(where={"title": "Discrete State Markov Chain Equilibrium"})   # no 'include' arg at all
print("zgomot docs:", len(res["ids"]))

zgomot docs: 30


## Verify Document Metadata

In [10]:
probe = coll.get(limit=5000)  # adjust as needed
metas  = [m for m in probe.get("metadatas", []) if m]
len(metas)

30

In [11]:
keys = set().union(*(m.keys() for m in metas)) if metas else set()
print("metadata keys:", sorted(keys))


metadata keys: ['authors', 'ext', 'filename', 'page', 'path', 'published_date', 'published_date_unix', 'section', 'source', 'tags', 'title', 'topic']


In [13]:
print("title counts:", Counter(m.get("title") for m in metas if "title" in m and m.get("title")).most_common(10))
print("published_date counts:", Counter(m.get("published_date") for m in metas if "published_date" in m and m.get("published_date")).most_common(10))

title counts: [('Discrete State Markov Chain Equilibrium', 30)]
published_date counts: [('2018-08-08', 30)]


## Search Validation

### Similarity

In [16]:
vs = DocumentLibraryLoader(DB_PATH).vectorstore

docs = vs.similarity_search("What is the definition of stochastic matrix", k=5, filter={"title": "Discrete State Markov Chain Equilibrium"})
for d in docs:
    print(d.metadata.get("path"), "—", d.metadata.get("section"))
    print(d.page_content[:250].replace("\n"," "), "\n")

./document_library/Discrete State Markov Chain Equilibrium.pdf — Page 18
both stochastic. Expressions were derived for the time evolution of any transition matrix and distribution and the equilibrium solutions were then obtained analytically by evaluating π  t t=100 (16) (9) 

./document_library/Discrete State Markov Chain Equilibrium.pdf — Page 2
3/31/19, 10(35 AMDiscrete State Markov Chain Equilibrium | gly.fish Page 2 of 19http://gly.fish.site.s3-website-us-east-1.amazonaws.com/2018/08/08/discrete_state_markov_chain_equilibrium.html  will be an  matrix where  is determined by the number of  

./document_library/Discrete State Markov Chain Equilibrium.pdf — Page 1
previous state  as defined for a Markov Process. At the next time step  the process state will be  with probability, since by definition the probability of state transition depends only upon the state at the previous time step. For an arbitrary time  

./document_library/Discrete State Markov Chain Equilibrium.pdf — Page 6


### MMR Search

In [ ]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 8, "fetch_k": 50, "filter": {"title": "Discrete State Markov Chain Equilibrium"}}
)

hits = retriever.invoke("What is the definition of stochastic matrix")
for d in hits[:5]:
    print(d.metadata.get("path"), d.metadata.get("section"))

./document_library/Discrete State Markov Chain Equilibrium.pdf Page 18
./document_library/Discrete State Markov Chain Equilibrium.pdf Page 2
./document_library/Discrete State Markov Chain Equilibrium.pdf Page 1
./document_library/Discrete State Markov Chain Equilibrium.pdf Page 6
./document_library/Discrete State Markov Chain Equilibrium.pdf Page 2


## Search Examples 

In [18]:
retriever = vs.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 8,
        "fetch_k": 60,
        "filter": {"$and": [{"title": "Discrete State Markov Chain Equilibrium"}, {"authors": "Troy Stribling"}]},
    },
)
hits = retriever.invoke("What is the definition of stochastic matrix")

[print(h) for h in hits]

page_content='both stochastic. Expressions were derived for the time evolution of any transition matrix
and distribution and the equilibrium solutions were then obtained analytically by evaluating
π 
t t=100 (16)
(9)' metadata={'published_date_unix': 1533686400, 'ext': '.pdf', 'topic': 'Discrete State Markov Chain Equilibrium', 'filename': 'Discrete State Markov Chain Equilibrium.pdf', 'authors': 'Troy Stribling', 'path': './document_library/Discrete State Markov Chain Equilibrium.pdf', 'published_date': '2018-08-08', 'page': 18, 'section': 'Page 18', 'title': 'Discrete State Markov Chain Equilibrium', 'tags': 'math,probability', 'source': 'pdf-library'}
page_content='3/31/19, 10(35 AMDiscrete State Markov Chain Equilibrium | gly.fish
Page 2 of 19http://gly.fish.site.s3-website-us-east-1.amazonaws.com/2018/08/08/discrete_state_markov_chain_equilibrium.html
 will be an  matrix where  is determined by the number of possible states. Each
row represents the Markov Chain transition probabil

[None, None, None, None, None, None, None, None]

### Use MMR Call Directly

In [19]:
docs = vs.max_marginal_relevance_search(
    "What is the definition of stochastic matrix",
    k=8,
    fetch_k=60,
    filter={"$and": [{"title": "Discrete State Markov Chain Equilibrium"}, {"authors": "Troy Stribling"}]},
)

[print(h) for h in hits]

page_content='both stochastic. Expressions were derived for the time evolution of any transition matrix
and distribution and the equilibrium solutions were then obtained analytically by evaluating
π 
t t=100 (16)
(9)' metadata={'published_date_unix': 1533686400, 'ext': '.pdf', 'topic': 'Discrete State Markov Chain Equilibrium', 'filename': 'Discrete State Markov Chain Equilibrium.pdf', 'authors': 'Troy Stribling', 'path': './document_library/Discrete State Markov Chain Equilibrium.pdf', 'published_date': '2018-08-08', 'page': 18, 'section': 'Page 18', 'title': 'Discrete State Markov Chain Equilibrium', 'tags': 'math,probability', 'source': 'pdf-library'}
page_content='3/31/19, 10(35 AMDiscrete State Markov Chain Equilibrium | gly.fish
Page 2 of 19http://gly.fish.site.s3-website-us-east-1.amazonaws.com/2018/08/08/discrete_state_markov_chain_equilibrium.html
 will be an  matrix where  is determined by the number of possible states. Each
row represents the Markov Chain transition probabil

[None, None, None, None, None, None, None, None]